# Modelo Yeast9

In [21]:
from cobra.io import read_sbml_model
from cobra.flux_analysis import flux_variability_analysis
import numpy as np

In [3]:
model_yeast9 = read_sbml_model("yeast-GEM.xml")

In [4]:
# Imprimir información básica del modelo
print("Reacciones:", len(model_yeast9.reactions))
print("Reacciones de exchange:", len(model_yeast9.exchanges))
print("Metabolitos:", len(model_yeast9.metabolites))
print("Genes:", len(model_yeast9.genes))

Reacciones: 4131
Reacciones de exchange: 271
Metabolitos: 2806
Genes: 1161


In [5]:
contador_exportaciones = 0
contador_importaciones = 0
for rxn in model_yeast9.exchanges:
    if rxn.lower_bound < 0:  # Solo reacciones de importación
        contador_importaciones += 1
        print(f"{rxn.id}: {rxn.name} (límite: {rxn.lower_bound})")
    if rxn.upper_bound > 0:  # Solo reacciones de exportación
        contador_exportaciones += 1
        
print(f"Total de reacciones de importación: {contador_importaciones}")
print(f"Total de reacciones de exportación: {contador_exportaciones}")

r_1654: ammonium exchange (límite: -1000.0)
r_1714: D-glucose exchange (límite: -1.0)
r_1832: H+ exchange (límite: -1000.0)
r_1861: iron(2+) exchange (límite: -1000.0)
r_1992: oxygen exchange (límite: -1000.0)
r_2005: phosphate exchange (límite: -1000.0)
r_2020: potassium exchange (límite: -1000.0)
r_2049: sodium exchange (límite: -1000.0)
r_2060: sulphate exchange (límite: -1000.0)
r_2100: water exchange (límite: -1000.0)
r_4593: chloride exchange (límite: -1000.0)
r_4594: Cu2(+) exchange (límite: -1000.0)
r_4595: Mn(2+) exchange (límite: -1000.0)
r_4596: Zn(2+) exchange (límite: -1000.0)
r_4597: Mg(2+) exchange (límite: -1000.0)
r_4600: Ca(2+) exchange (límite: -1000.0)
Total de reacciones de importación: 16
Total de reacciones de exportación: 270


In [22]:
def imprimir_precios_sombra(solucion, model):
    for met_id, value in solucion.shadow_prices.items():
        if abs(value) > 1:
            met = model.metabolites.get_by_id(met_id)
            print(f"{met_id} | {met.name} | shadow price = {value:.3f}")
            
            
def imprimir_costos_reducidos(solucion, model):
    print("Reacciones con costos reducidos positivos")
    # Imprimir resultados
    for rxn_id, value in solucion.reduced_costs.items():
        if value > 1e-6 and model.reactions.get_by_id(rxn_id) in model.exchanges:
            rxn = model.reactions.get_by_id(rxn_id)
            print(f"{rxn_id} | {rxn.name} | shadow price = {value:.3f}")
            
    print("\nReacciones con costos reducidos negativos")
    # Imprimir resultados
    for rxn_id, value in solucion.reduced_costs.items():
        if value < -1 and model.reactions.get_by_id(rxn_id) in model.exchanges:
            rxn = model_yeast9.reactions.get_by_id(rxn_id)
            print(f"{rxn_id} | {rxn.name} | shadow price = {value:.3f}")

Los precios sombra son por metabolito. 

Si el precio sombra de un metabolito es alto (en valor absoluto), significa que: 
- Ese metabolito es limitante para el crecimiento. 
- Una pequeña disponibilidad extra mejoraría la biomasa.

Los costos reducidos son cantidades asociadas a cada reacción (flujo) que indican cuánto tendría que mejorar esa reacción en términos del objetivo (por ejemplo, biomasa) para que pase de estar inactiva (flujo cero) a activarse en la solución óptima. 

En términos prácticos, si estás maximizando crecimiento y una reacción tiene reduced cost positivo:
-  activarla aumentaría la biomasa, pero actualmente está impedida por alguna restricción (como un bound cerrado).

si es negativo:
- activarla disminuiría la biomasa y por eso el modelo no la usa.

y si es cero:
- la reacción ya es óptima o indiferente en la solución actual. 

En resumen, los reduced costs te dicen qué reacciones “querrían” activarse para mejorar el objetivo y cuáles están correctamente apagadas bajo las condiciones actuales.

## FBA modelo tal cual:

In [24]:
with model_yeast9:
    sol1 = model_yeast9.optimize()
    print(f"Solución optimización: {sol1.objective_value}")
    fva_res = flux_variability_analysis(model_yeast9, model_yeast9.exchanges, fraction_of_optimum=0.9)

Solución optimización: 0.08584393900351904


In [25]:
print(fva_res)

        minimum   maximum
r_1542      0.0  0.000000
r_1545      0.0  0.000000
r_1546      0.0  0.174519
r_1547      0.0  0.083836
r_1548      0.0  0.042558
...         ...       ...
r_4737      0.0  0.116112
r_4739      0.0  0.000000
r_4741      0.0  0.077299
r_4743      0.0  0.063647
r_4780      0.0  0.008785

[271 rows x 2 columns]


In [8]:
imprimir_precios_sombra(sol1, model_yeast9)

s_0045 | (S)-3-hydroxyhexacosanoyl-CoA | shadow price = -1.091
s_0243 | 3-oxohexacosanoyl-CoA | shadow price = -1.085
s_0672 | ergosterol ester backbone | shadow price = -1.386
s_0816 | hexacosanoyl-CoA | shadow price = -1.086
s_0817 | hexacosanoyl-CoA | shadow price = -1.086
s_0819 | hexacosanoyl-CoA | shadow price = -1.091
s_0894 | inositol-P-ceramide A (C24) | shadow price = -1.004
s_0895 | inositol-P-ceramide A (C24) | shadow price = -1.004
s_0897 | inositol-P-ceramide A (C26) | shadow price = -1.039
s_0898 | inositol-P-ceramide A (C26) | shadow price = -1.039
s_0900 | inositol-P-ceramide B (C24) | shadow price = -1.004
s_0901 | inositol-P-ceramide B (C24) | shadow price = -1.004
s_0903 | inositol-P-ceramide B (C26) | shadow price = -1.039
s_0904 | inositol-P-ceramide B (C26) | shadow price = -1.039
s_1479 | tetracosanoyl-CoA | shadow price = -1.051
s_1480 | tetracosanoyl-CoA | shadow price = -1.051
s_1482 | tetracosanoyl-CoA | shadow price = -1.055
s_1513 | trans-hexacos-2-enoyl-C

In [9]:
imprimir_costos_reducidos(sol1, model_yeast9)

Reacciones con costos reducidos positivos

Reacciones con costos reducidos negativos
r_1753 | episterol exchange | shadow price = -1.562
r_1757 | ergosterol exchange | shadow price = -1.606
r_1788 | fecosterol exchange | shadow price = -1.562
r_1915 | lanosterol exchange | shadow price = -1.388
r_2106 | zymosterol exchange | shadow price = -1.519
r_2134 | 14-demethyllanosterol exchange | shadow price = -1.436
r_2137 | ergosta-5,7,22,24(28)-tetraen-3beta-ol exchange | shadow price = -1.591
r_4780 | heme a exchange | shadow price = -1.916


## FBA modelo con glucosa en -1000

In [ ]:
with model_yeast9:
    model_yeast9.exchanges.get_by_id("r_1714").lower_bound = -1000.0
    sol2 = model_yeast9.optimize()
    print(f"Solución optimización: {sol2.objective_value}")
    fva_res2 = flux_variability_analysis(model_yeast9, model_yeast9.exchanges, fraction_of_optimum=0.9)

Solución optimización: 20.540587160229773


In [ ]:
imprimir_precios_sombra(sol2, model_yeast9)

s_3234 | cardiolipin (1-16:0, 2-16:1, 3-16:0, 4-16:1) | shadow price = -1.088
s_3235 | cardiolipin (1-16:0, 2-16:1, 3-16:1, 4-16:1) | shadow price = -1.080
s_3236 | cardiolipin (1-16:0, 2-16:1, 3-18:0, 4-16:1) | shadow price = -1.123
s_3237 | cardiolipin (1-16:0, 2-16:1, 3-18:1, 4-16:1) | shadow price = -1.115
s_3238 | cardiolipin (1-16:0, 2-16:1, 3-16:0, 4-18:1) | shadow price = -1.124
s_3239 | cardiolipin (1-16:0, 2-16:1, 3-16:1, 4-18:1) | shadow price = -1.115
s_3240 | cardiolipin (1-16:1, 2-16:1, 3-16:0, 4-16:1) | shadow price = -1.080
s_3241 | cardiolipin (1-16:1, 2-16:1, 3-16:1, 4-16:1) | shadow price = -1.071
s_3242 | cardiolipin (1-16:1, 2-16:1, 3-18:0, 4-16:1) | shadow price = -1.115
s_3243 | cardiolipin (1-16:1, 2-16:1, 3-18:1, 4-16:1) | shadow price = -1.107
s_3244 | cardiolipin (1-16:1, 2-16:1, 3-16:0, 4-18:1) | shadow price = -1.115
s_3245 | cardiolipin (1-16:1, 2-16:1, 3-16:1, 4-18:1) | shadow price = -1.107
s_3246 | cardiolipin (1-18:0, 2-16:1, 3-16:0, 4-16:1) | shadow p

In [16]:
imprimir_costos_reducidos(sol2, model_yeast9)

Reacciones con costos reducidos positivos

Reacciones con costos reducidos negativos
r_1753 | episterol exchange | shadow price = -1.562
r_1757 | ergosterol exchange | shadow price = -1.606
r_1788 | fecosterol exchange | shadow price = -1.562
r_1915 | lanosterol exchange | shadow price = -1.388
r_2106 | zymosterol exchange | shadow price = -1.519
r_2134 | 14-demethyllanosterol exchange | shadow price = -1.436
r_2137 | ergosta-5,7,22,24(28)-tetraen-3beta-ol exchange | shadow price = -1.591
r_4780 | heme a exchange | shadow price = -1.916
